In [2]:
import pandas as pd

In [15]:
FOLDER = r"C:\Users\samee\Documents\CHIEAC\HOPE_WSDM_2022\Train"

In [10]:
import pandas as pd

def dataframe_to_dialogue_txt(df: pd.DataFrame) -> None:
    """
    df columns expected:
      - Type: 'P' for Patient, 'T' for Therapist (or similar)
      - Utterance: text
    Writes a text file where consecutive same-speaker utterances are concatenated.
    """

    # Keep original order (important). If you have a turn/timestamp column, sort by it here.
    df = df.reset_index(drop=True).copy()

    # Clean utterances a bit (optional but helpful)
    df["Utterance"] = df["Utterance"].fillna("").astype(str).str.strip()

    # 1) Start a new chunk whenever speaker changes
    df["chunk_id"] = (df["Type"] != df["Type"].shift(1)).cumsum()

    # 2) Concatenate utterances within each chunk, preserving order
    chunks = (
        df.groupby(["chunk_id", "Type"], as_index=False)["Utterance"]
          .apply(lambda s: " ".join([x for x in s if x]))  # join non-empty
          .rename(columns={"Utterance": "combined"})
    )

    # 3) Map Type -> label
    speaker_map = {"P": "Patient", "T": "Therapist"}

    # 4) Write to file
    lines = []
    for _, row in chunks.iterrows():
        label = speaker_map.get(row["Type"], str(row["Type"]))
        text = row["combined"]
        if text:  # skip empty
            lines.append(f"<{label}>: {text}")

    return lines

In [18]:
import glob
files = glob.glob(FOLDER+"/*.csv")

In [20]:
for i, file in enumerate(files):
    df = pd.read_csv(file)
    lines = dataframe_to_dialogue_txt(df)
    with open(f"{FOLDER}/Transcript_{i+1}.txt", "w", encoding="utf-8") as f:
        f.write("\n\n".join(lines))
    